In [1]:
import duckdb
import requests
import pandas as pd

In [2]:
conn = duckdb.connect("v2_database.duckdb")
df_gen = conn.execute("select * from generations").fetch_df()
conn.close()

# Filter the first row where 'status' is blank (NaN or empty string)
run_gen = df_gen[df_gen['STATUS'].isna() | (df_gen['STATUS'] == '')].iloc[0]['NAME']

# Show value 'name'
print(f'Running for Gen: {run_gen}')

CatalogException: Catalog Error: Table with name generations does not exist!
Did you mean "pg_constraint"?
LINE 1: select * from generations
                      ^

In [34]:
# URL da API para a geração específica
url = f"https://pokeapi.co/api/v2/generation/{run_gen}/"

# Requisição para obter os dados da geração
response = requests.get(url)

# Verificar se a requisição foi bem-sucedida
if response.status_code == 200:
    data = response.json()
    
    # Extrair a lista de Pokémon da geração
    pokemon_species = data.get("pokemon_species", [])
    
    # Extrair o nome e o ID do Pokémon a partir da URL
    pokemon_data = [
        {
            "name": pokemon["name"],
            "id": int(pokemon["url"].rstrip("/").split("/")[-1])  # Extrair o ID da URL
        }
        for pokemon in pokemon_species
    ]
    
    # Criar o DataFrame
    df_pokemon = pd.DataFrame(pokemon_data)
    # Ordenar o DataFrame pela coluna 'id' em ordem crescente
    df_pokemon = df_pokemon.sort_values(by='id', ascending=True)

    # Obter o número de linhas e colunas do DataFrame
    num_linhas, num_colunas = df_pokemon.shape

    # Exibir o resultado
    print(f"Dataframe has {num_linhas} rows and {num_colunas} columns.")
    
else:
    print(f"Erro ao acessar a API: {response.status_code}")

Dataframe has 151 rows and 2 columns.


In [35]:
conn = duckdb.connect("v2_database.duckdb")
conn.execute("""
    CREATE TABLE IF NOT EXISTS b_pokemons ( 
      NAME TEXT,
      ID TEXT
    )
""")
conn.execute("""INSERT INTO b_pokemons SELECT * FROM df_pokemon""")
conn.close()